## Configuration


In [0]:

class Config():
    def __init__(self):
        self.db_name = "workout"
        self.max_FilesPerTrigger = 100

In [0]:
config = Config()
print(f"Database Name: {config.db_name}")

## Set Up


In [0]:
import os
def get_spark():
    """Get Spark session based on environment"""
    
    if "DATABRICKS_RUNTIME_VERSION" in os.environ:
        print("Running in Databricks environment")
        from databricks.sdk.runtime import spark
        return spark
    else:
        print("Running locally with Databricks Connect")
        from databricks.connect import DatabricksSession
        spark = (DatabricksSession
                .builder
                .profile("dev-free-edition")
                .serverless(True)
                .getOrCreate())
        return spark


spark = get_spark()
spark

In [0]:

class SetUp():
    def __init__(self,env):
        conf = Config()
        self.db_name = conf.db_name
        self.catalog = env
        self.initialized = False


    def create_db(self):
        print(f"Creating the database {self.catalog}.{self.db_name}...", end='')
        spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.catalog}.{self.db_name}")
        spark.sql(f"USE {self.catalog}.{self.db_name}")
        self.initialized = True
        print("Done.")













In [0]:
setup = SetUp(env="dev")


In [0]:
setup.create_db()

In [0]:
%sql

show databases in dev;

In [0]:

def get_secrets(name: str,  scope: str = "test", env: str = "local"):
    """Get secrets from environment variables or Databricks Secrets"""

    if env == "local":

        from dotenv import load_dotenv
        import os

        load_dotenv()
        return os.getenv(name)
    
    try:
        return dbutils.secrets.get(scope=scope, key=name)
    except Exception as e:
        print(f"Error getting secret {name} from scope {scope}: {e}")

    raise ValueError(f"Secret {name} not found in scope {scope}")


In [0]:
azure_storage_account = get_secrets("azure_storage_account")


In [0]:
catalog = "dev"

ingestion_db = "control"
ingestion_table = "ingest_table_registry"
bronze_db = "bronze_workout"


# spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.{ingestion_db};")

# spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.{bronze_db};")

In [0]:
query = f"""

CREATE TABLE IF NOT EXISTS {catalog}.{ingestion_db}.{ingestion_table} (
  table_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  source_path STRING NOT NULL,
  source_format STRING DEFAULT 'parquet',
  header BOOLEAN DEFAULT TRUE,
  table_version INT DEFAULT 1,

  target_catalog STRING DEFAULT 'dev',
  target_schema STRING DEFAULT 'test',
  target_table STRING NOT NULL,
  write_mode STRING DEFAULT 'append'
      CHECK (write_mode IN ('append', 'merge', 'overwrite')),
  checkpoint_path STRING NOT NULL,
  merge_key ARRAY<STRING> DEFAULT ARRAY(),
  partition_by ARRAY<STRING> DEFAULT ARRAY(),
  enabled BOOLEAN DEFAULT TRUE,

 trigger_mode STRING DEFAULT 'availableNow'
    CHECK (trigger_mode IN ('availableNow', 'processingTime', 'continuous')),
  processing_time STRING DEFAULT '5 minutes',
  schema_evolution_mode STRING DEFAULT 'BACKWARD'
    CHECK (schema_evolution_mode IN ('BACKWARD', 'FORWARD', 'FULL', 'NONE')),
  
  expected_rows_per_day BIGINT,
  last_success TIMESTAMP,
  last_error STRING,
  consecutive_failures INT,
  last_run_duration_seconds INT,
  created_by STRING,
  created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  updated_by STRING,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.autoCompact' = 'true',
  'delta.feature.allowColumnDefaults' = 'supported'
);

"""

In [0]:
spark.sql(query)

In [0]:
insert_query = f"""

INSERT INTO {catalog}.{ingestion_db}.{ingestion_table} 
(source_path,                         source_format,header,table_version,target_catalog,target_schema,target_table,write_mode,checkpoint_path,merge_key,partition_by,enabled,trigger_mode,processing_time,schema_evolution_mode,expected_rows_per_day,last_success,last_error,consecutive_failures,last_run_duration_seconds,created_by,created_at,updated_by,updated_at)
VALUES 
('/Volumes/dev/bronze_workout/rw/bpm/', 'json', TRUE, 1, 'dev', 'bronze_workout', 'bpm', 'append', '/Volumes/dev/bronze_workout/checkpoints/', ARRAY(), ARRAY(), TRUE, 'availableNow', '5 minutes', 'BACKWARD', 1000000, NULL, NULL, 0, NULL, 'gaurav.thagunna', CURRENT_TIMESTAMP(), NULL, CURRENT_TIMESTAMP());

"""

spark.sql(insert_query)

In [0]:
configs = spark.sql(f"select * from {catalog}.{ingestion_db}.{ingestion_table}")
configs.show(truncate=False)

In [0]:
configs_dict = configs.select("table_id","source_path","source_format","header","target_catalog","target_schema","table_version","target_table","write_mode","checkpoint_path","merge_key","partition_by","enabled","trigger_mode","processing_time","schema_evolution_mode").collect()[0].asDict()
configs_dict

In [0]:
def get_schema_file(path:str):
    df = spark.read.json(path)
    return df.schema


In [0]:
schema = get_schema_file(path=configs_dict["source_path"])
schema

In [0]:
ingestion_db = "control"
schema_registry_tb = "schema_registry"

schema_registry_query = f"""CREATE TABLE IF NOT EXISTS {catalog}.{ingestion_db}.{schema_registry_tb} (
  id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  table_id INT NOT NULL,
  table_name STRING NOT NULL,
  version INT DEFAULT 1,
  is_latest BOOLEAN DEFAULT TRUE,                 
  schema_format STRING,          
  schema_definition STRING,     
  created_by STRING,
  created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP()
  )
  TBLPROPERTIES (
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.feature.allowColumnDefaults' = 'supported'
  );

  """

spark.sql(schema_registry_query)

In [0]:
spark.table(f"{catalog}.{ingestion_db}.{schema_registry_tb}").show()

In [0]:
spark.sql(f"SELECT * FROM {catalog}.{ingestion_db}.{schema_registry_tb}").show(truncate=False)

In [0]:
query = f"""

INSERT INTO {catalog}.{ingestion_db}.{schema_registry_tb} (
  table_id,
  table_name,
  version,
  is_latest,
  schema_format,
  schema_definition,
  created_by
)
VALUES (
  {configs_dict["table_id"]},
  '{configs_dict["target_table"]}',
  1,
  TRUE,
  'json',
  '{schema.json()}',
  'gaurav.thagunna'
)

"""

spark.sql(query)



In [0]:
spark.table("dev.control.schema_registry").show(truncate=False)

In [0]:
schema_row = (
    spark.table("dev.control.schema_registry")
         .filter("table_id = 1 AND is_latest = TRUE").limit(1).collect()[0]


)
schema_row

In [0]:
from pyspark.sql.types import StructType
import json
schema = StructType.fromJson(json.loads(schema_row.schema_definition))
schema

In [0]:
spark.read.json(configs_dict["source_path"]).show(truncate=False)

In [0]:

spark.sql(f"create table dev.bronze_workout.test")

In [0]:

df = spark.read.schema(schema).json(configs_dict["source_path"])
df.show(truncate=False)

In [0]:
df.printSchema()

In [0]:
configs_dict

In [0]:
from pyspark.sql import SparkSession, DataFrame

def load_date(spark, configs_dict,schema) -> DataFrame:
    """Load data from a given path with the provided schema."""

    return (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", configs_dict["source_format"])
            .option("cloudFiles.schemaLocation", configs_dict["checkpoint_path"]+"schema/"+configs_dict["target_table"])
            .option("cloudFiles.schemaEvolutionMode","rescue")
            .option("cloudFiles.maxFilesPerTrigger", 1)
            .schema(schema)
            .load(configs_dict["source_path"])

    )


In [0]:
raw_data = load_date(spark=spark,configs_dict=configs_dict,schema=schema)


In [0]:
def view_streaming_df(df):
    """View streaming DataFrame"""
    return display(
        df,
        checkpointLocation=f"{configs_dict['checkpoint_path']}tmp/{configs_dict['target_table']}"
    )


In [0]:
view_streaming_df(df=raw_data)

In [0]:
def ensure_target_table_exists(spark,configs_dict):
    try:
        print(f"creating table")
        spark.sql(f"DESCRIBE TABLE {configs_dict['target_catalog']}.{configs_dict['target_schema']}.{configs_dict['target_table']}")
        return
    except Exception:
        pass

    spark.sql(f"CREATE TABLE {configs_dict['target_catalog']}.{configs_dict['target_schema']}.{configs_dict['target_table']}")







In [0]:
ensure_target_table_exists(spark,configs_dict)

In [0]:
configs_dict

In [0]:
type(configs_dict)

In [0]:
def merge_to_delta(spark: SparkSession,df: DataFrame,configs_dict):

    ensure_target_table_exists(spark,configs_dict)    
    print(f"merge keys : {configs_dict['merge_key']}")

    target_full_name = (f"{configs_dict['target_catalog']}."
                        f"{configs_dict['target_schema']}."
                        f"{configs_dict['target_table']}")
    
    if not configs_dict["merge_key"]:
        print("merge keys not present")
        return (df.write
            .format("delta") 
            .option("mergeSchema", "true") 
            .mode(configs_dict["write_mode"]) 
            .saveAsTable(target_full_name)
            )

    else:
        df.createOrReplaceTempView("source_table") 

        condition = " AND ".join([f"t.{key} = s.{key}" for key in configs_dict['merge_key']])

        query = f"""
        MERGE INTO {target_full_name} AS t
        USING source_table AS s
        ON {condition}
        WHEN MATCHED THEN
        UPDATE SET * 
        WHEN NOT MATCHED THEN
        INSERT * 
            
        """

        spark.sql(query)


In [0]:
def start_stream(df, configs_dict):
    writer = (
        df.writeStream
          .foreachBatch(lambda batch_df, batch_id:
                        merge_to_delta(spark, batch_df, configs_dict))
         .option("checkpointLocation", configs_dict["checkpoint_path"]+configs_dict["target_table"]) 
          .queryName(configs_dict["target_table"])
    )

    if configs_dict["trigger_mode"] == "availableNow":
        return writer.trigger(availableNow=True).start()
    else:
        return writer.trigger(processingTime=configs_dict["processing_time"]).start()


In [0]:
start_stream(df=raw_data, configs_dict=configs_dict)

In [0]:
rows = spark.table("dev.control.ingest_table_registry").filter("enabled = true").collect()
rows


In [0]:
tables = [row.asDict() for row in rows]
tables

In [0]:
history_table = "ingest_run_history"
query = f"""
CREATE TABLE IF NOT EXISTS {configs_dict["target_catalog"]}.{configs_dict["ingestion_db"]}.{history_table} (
    
  id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  run_id STRING NOT NULL,
  table_id INT NOT NULL,
  start_ts TIMESTAMP,          
  status STRING,     
  created_by STRING,
  created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP()
  )
  TBLPROPERTIES (
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.feature.allowColumnDefaults' = 'supported'
  );    

"""

In [0]:
import json, time, datetime, uuid

def now_ts():
    return datetime.datetime.utcnow().isoformat()

def start_run(table_name: str) -> str:
    run_id = str(uuid.uuid4())
    start_ts = now_ts()
    spark.sql(f"""INSERT INTO {configs_dict['target_catalog']}.ingest_run_history (run_id, table_name, start_ts, status, created_by, created_at)
                  VALUES ('{run_id}','{table_name}', TIMESTAMP'{start_ts}', 'running', 'ingest_worker', TIMESTAMP'{start_ts}')""")
    return run_id



In [0]:

def finish_run_success(run_id, table_name, duration_seconds, rows_in=None, rows_out=None):
    end_ts = now_ts()
    spark.sql(f"""UPDATE control.ingest_run_history SET end_ts = TIMESTAMP'{end_ts}', status='success', duration_seconds={duration_seconds}, rows_in={rows_in or 'NULL'}, rows_out={rows_out or 'NULL'} WHERE run_id='{run_id}'""")
    spark.sql(f"""MERGE INTO control.ingest_table_registry tr USING (SELECT '{table_name}' as table_name) s ON tr.table_name = s.table_name WHEN MATCHED THEN UPDATE SET last_success = TIMESTAMP'{end_ts}', last_error = NULL, consecutive_failures = 0, last_run_duration_seconds = {duration_seconds}""")

def finish_run_failure(run_id, table_name, duration_seconds, err_msg):
    end_ts = now_ts()
    esc = err_msg.replace("'", "''")[:2000]
    spark.sql(f"""UPDATE control.ingest_run_history SET end_ts = TIMESTAMP'{end_ts}', status='failed', error = '{esc}', duration_seconds = {duration_seconds} WHERE run_id = '{run_id}'""")
    spark.sql(f"""MERGE INTO control.ingest_table_registry tr USING (SELECT '{table_name}' as table_name) s ON tr.table_name = s.table_name WHEN MATCHED THEN UPDATE SET last_error = '{esc}', consecutive_failures = coalesce(tr.consecutive_failures,0)+1, last_run_duration_seconds = {duration_seconds}""")


In [0]:
CREATE TABLE control.bronze_tables (
  table_name STRING,
  source_path STRING,
  format STRING,
  header BOOLEAN,
  schema_id STRING,
  merge_key STRING,
  partition_by STRING,
  enabled BOOLEAN,
  trigger_mode STRING,       
  processing_time STRING,  
  owner STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA;

In [0]:
!pwd